# Tier 0 checks: is the BHLUMI sample right, and does it reach WarpX intact?

This notebook walks through the Tier 0 tests of the compatibility test plan, one picture per test. None of them
runs WarpX: they use the BHLUMI event sample, the conversion code in `wx_bhabha.py`, and its vertex sampler.

Each section says **what is checked**, **what you should see** in the plot, and ends with a PASS / FAIL line using
the same criteria as `tests/tier0_checks.py`. A summary table is at the bottom.

Colours: **blue = what the code produced**, **orange = what physics predicts**, grey = reference lines.

In [ ]:
import math, os, sys
import numpy as np
import matplotlib.pyplot as plt

REPO = os.environ.get("BHLUMI_WARPX", os.path.abspath(".."))      # the notebook lives in <repo>/notebooks
sys.path[:0] = [REPO, os.path.join(REPO, "tests"), os.path.join(REPO, "benchmarks")]
import tier0_checks as t0          # helper functions shared with the command-line checks
import wx_bhabha as wx             # the code that hands BHLUMI events to WarpX

PS = os.environ.get("PSCRATCH", "/pscratch/sd/a/alkinder")
SAMPLE  = os.environ.get("T0_SAMPLE",  f"{PS}/bhlumi/fcc_z/seed_2")   # 10^6-event sample
SAMPLE2 = os.environ.get("T0_SAMPLE2", f"{PS}/bhlumi/fcc_z/seed_3")   # optional second seed (test A8)
RUN_DIR = os.environ.get("T0_RUN", f"{PS}/runs/warpx/bhabha_benchmark/dbg_blocking/seed_1")  # optional (C7)
HEADON  = os.path.join(REPO, "benchmarks/headon_round/inputs_headon_round.txt")
FCC     = os.path.join(REPO, "examples/fcc_z_boxscan_template.txt")
TH1, TH2 = 0.040, 0.120            # generation window of the card [rad]

BLUE, ORANGE, GREY, INK = "#2a78d6", "#eb6834", "#8a8984", "#52514e"
plt.rcParams.update({
    "figure.figsize": (7.5, 3.6), "figure.dpi": 110, "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": INK,
    "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
    "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.8,
    "lines.linewidth": 2, "legend.frameon": False, "axes.titlesize": 12, "axes.titleweight": "regular",
})
SUMMARY = []
def verdict(test, ok, msg):
    SUMMARY.append((test, ok, msg))
    print(("PASS" if ok else "FAIL") + f"  {test}: {msg}")

ev = np.load(os.path.join(SAMPLE, "events.npz"))
ele, pos, ph, ph_evt = ev["ele"], ev["pos"], ev["ph"], ev["ph_evt"]
n = len(ele)
tot = t0.event_sums(ele.copy(), pos, ph, ph_evt)          # sum of all final-state four-momenta per event
sqrt_s = float(np.median(tot[:, 3])); s = sqrt_s ** 2; EBEAM = sqrt_s / 2
te, tp = t0.polar(ele, -1), t0.polar(pos, +1)              # angle of each lepton from its own beam
pair = ele + pos
v = 1 - (pair[:, 3] ** 2 - (pair[:, :3] ** 2).sum(1)) / s  # v = 1 - s'/s: fraction of sqrt(s)^2 lost to photons
print(f"{n} events, {len(ph)} photons, sqrt(s) = {sqrt_s:.4f} GeV, sigma = {float(ev['xsec_nb']):.4g} nb")

## Part A: the generator

### A2. Cross section

At small angles, Bhabha scattering is photon exchange between the two leptons, and the lowest-order (Born)
cross section in a polar-angle window [θ₁, θ₂] is

$$\sigma_\text{Born} = \frac{16\pi\alpha^2}{s}\left(\frac{1}{\theta_1^2}-\frac{1}{\theta_2^2}\right)(\hbar c)^2 .$$

**You should see** BHLUMI's cross section (blue) within a few percent of Born (orange). The difference is the
radiative correction, which BHLUMI includes and Born does not.

In [ ]:
xs, rel = float(ev["xsec_nb"]), float(ev["rel_err"])
born = 16 * math.pi * t0.ALPHA**2 / s * (1 / TH1**2 - 1 / TH2**2) * t0.HBARC2_NB

fig, ax = plt.subplots(figsize=(7.5, 1.9))
ax.axvspan(born * 0.95, born * 1.05, color=ORANGE, alpha=0.10, lw=0)
ax.axvline(born, color=ORANGE); ax.text(born, 1.32, " Born", color=ORANGE, va="center")
ax.errorbar([xs], [1], xerr=[xs * rel], fmt="o", color=BLUE, ms=8, capsize=4)
ax.text(xs, 0.68, f"BHLUMI {xs:.2f} nb ", color=INK, ha="right", va="center")
ax.set_ylim(0.4, 1.6); ax.set_yticks([]); ax.set_xlabel("cross section in 40–120 mrad [nb]")
ax.set_title(f"BHLUMI is {100 * (xs / born - 1):+.2f} % from Born (band: ±5 %)", loc="left")
plt.show()
verdict("A2", abs(xs / born - 1) < 0.05, f"{xs:.4g} nb vs Born {born:.4g} nb")

### A3. Angular distribution

Born scattering falls like dN/dθ ∝ θ⁻³: most leptons are at the small-angle edge. The clearest way to see this is
to divide the measured distribution by the Born shape. **A flat ratio means the shape is exactly Born.**

Radiation changes the shape slightly, in a predictable way. Keeping only near-elastic events (photons carry less
than 1 % of s, v < 0.01), the probability of *not* radiating falls slowly with angle, so the ratio should tilt
down like θ^Δn with Δn = (8α/π) ln 0.01 = −0.086 (orange). The full sample (grey) tilts the other way, because
very hard radiation moves leptons around; that part is not Born and is not tested.

In [ ]:
edges = np.geomspace(TH1, TH2, 16); mid = np.sqrt(edges[1:] * edges[:-1])
born_frac = np.diff(-1 / edges**2) / (1 / TH1**2 - 1 / TH2**2)          # Born fraction of leptons per bin

def ratio(sel):
    th = np.concatenate([te[sel], tp[sel]]); th = th[(th >= TH1) & (th <= TH2)]
    h = np.histogram(th, edges)[0]
    return h / h.sum() / born_frac, np.sqrt(h) / h.sum() / born_frac

def slope(sel):
    th = np.concatenate([te[sel], tp[sel]]); th = th[(th >= TH1) & (th <= TH2)]
    k = np.linspace(-4, -2, 4001)
    ll = k * np.log(th).sum() - len(th) * np.log((TH2**(k + 1) - TH1**(k + 1)) / (k + 1))
    i = np.argmax(ll); curv = (ll[i + 1] - 2 * ll[i] + ll[i - 1]) / (k[1] - k[0])**2
    return k[i], 1 / math.sqrt(-curv)

r_el, e_el = ratio(v < 0.01); r_all, e_all = ratio(np.ones(n, bool))
dn = 8 * t0.ALPHA / math.pi * math.log(0.01)
pred = mid**dn; pred /= (pred * born_frac).sum()

fig, ax = plt.subplots()
ax.axhline(1, color=GREY, lw=1, ls="--"); ax.text(TH2 * 1.01, 1, "Born", color=GREY, va="center")
ax.plot(mid, pred, color=ORANGE, label="prediction for v < 0.01")
ax.errorbar(mid, r_el, e_el, fmt="o", color=BLUE, ms=5, label="BHLUMI, near-elastic (v < 0.01)")
ax.errorbar(mid, r_all, e_all, fmt="o", color=GREY, ms=4, mfc="white", label="BHLUMI, all events")
ax.set_xscale("log"); ax.set_xticks([0.04, 0.06, 0.08, 0.1, 0.12]); ax.set_xticklabels(["40", "60", "80", "100", "120"])
ax.minorticks_off(); ax.set_xlabel("lepton polar angle θ [mrad]"); ax.set_ylabel("data / Born shape")
ax.set_title("Near-elastic events follow Born, tilted as radiation predicts", loc="left"); ax.legend(fontsize=9)
plt.show()

ok = True
for vc in (0.01, 0.1):
    k, dk = slope(v < vc); p = -3 + 8 * t0.ALPHA / math.pi * math.log(vc); ok &= abs(k - p) < 0.05
    print(f"   v < {vc:g}: dN/dθ ∝ θ^n with n = {k:.3f} ± {dk:.3f}, predicted {p:.3f}")
verdict("A3", ok, "near-elastic slopes within 0.05 of the prediction")

### A4. Azimuth

Nothing in the collision prefers a direction around the beam, so the azimuth φ of the scattered electron must be
uniform. **You should see** the histogram scattered around the flat line, inside the ±1σ band on most bins.

In [ ]:
nb = 36; h, ed = np.histogram(t0.azimuth(ele), nb, (-math.pi, math.pi)); mu = n / nb
fig, ax = plt.subplots()
ax.fill_between([-math.pi, math.pi], mu - math.sqrt(mu), mu + math.sqrt(mu), color=ORANGE, alpha=0.15, lw=0, label="flat ±1σ")
ax.axhline(mu, color=ORANGE, label="flat (expected)")
ax.stairs(h, ed, color=BLUE, lw=2, label="BHLUMI e⁻")
ax.set_ylim(mu * 0.97, mu * 1.03); ax.set_xlabel("azimuth φ [rad]"); ax.set_ylabel("electrons per bin")
chi2 = ((h - mu)**2 / mu).sum() / (nb - 1)
ax.set_title(f"Uniform in φ (χ²/ndf = {chi2:.2f})", loc="left"); ax.legend(fontsize=9, ncol=3, loc="upper left")
plt.show()
verdict("A4", chi2 < 2, f"chi2/ndf = {chi2:.2f}")

### A5. Back-to-back topology

Without a hard photon, momentum conservation forces the two leptons to leave back to back: the same polar angle
from their own beams, and opposite azimuths. **You should see** events with no photon above 1 GeV sitting on the
diagonal (left). Events with a hard photon (right) are allowed to spread, because the photon carries momentum
away.

In [ ]:
emax = np.zeros(n)
if len(ph): np.maximum.at(emax, ph_evt, ph[:, 3])
soft = emax < 1.0
rng = np.random.default_rng(0)
fig, axs = plt.subplots(1, 2, figsize=(8, 3.9), sharex=True, sharey=True)
for ax, sel, title in ((axs[0], soft, "no photon above 1 GeV"), (axs[1], ~soft, "with a photon above 1 GeV")):
    idx = np.flatnonzero(sel); idx = rng.choice(idx, min(4000, len(idx)), replace=False) if len(idx) else idx
    ax.plot([30, 140], [30, 140], color=ORANGE, lw=1.5)
    ax.scatter(1e3 * te[idx], 1e3 * tp[idx], s=3, color=BLUE, alpha=0.4, lw=0)
    ax.set_title(f"{title} ({100 * sel.mean():.0f} %)", loc="left", fontsize=11)
    ax.set_xlim(30, 140); ax.set_ylim(30, 140); ax.set_xlabel("θ of e⁻ [mrad]")
axs[0].set_ylabel("θ of e⁺ [mrad]")
plt.tight_layout(); plt.show()

acop = math.pi - np.abs(np.angle(np.exp(1j * (t0.azimuth(ele[soft]) - t0.azimuth(pos[soft])))))
dth = np.abs(te[soft] - tp[soft]) / te[soft]
qa, qd = np.quantile(acop, 0.99), np.quantile(dth, 0.99)
verdict("A5", qa < 0.05 and qd < 0.05, f"99 % of soft events: acoplanarity < {1e3 * qa:.2f} mrad, |Δθ|/θ < {qd:.4f}")

### A6. Lepton energies

A lepton can lose energy to photons but can never gain any, so no lepton may exceed the beam energy (45.6 GeV).
**You should see** a sharp peak at the beam energy, a radiative tail to the left, and nothing to the right of the
orange line. The y axis is logarithmic so the rare low-energy leptons are visible.

In [ ]:
El = np.concatenate([ele[:, 3], pos[:, 3]])
fig, ax = plt.subplots()
ax.hist(El, bins=np.linspace(0, EBEAM * 1.05, 120), color=BLUE, histtype="stepfilled", alpha=0.85)
ax.axvline(EBEAM, color=ORANGE)
ax.set_yscale("log"); ax.set_xlabel("lepton energy E′ [GeV]"); ax.set_ylabel("leptons per bin")
ax.annotate("beam energy", (EBEAM, 0.5), xycoords=("data", "axes fraction"), xytext=(-8, 0),
            textcoords="offset points", ha="right", color=ORANGE)
peak, above = np.mean(np.abs(El / EBEAM - 1) < 0.01), np.mean(El > EBEAM * (1 + 1e-6))
ax.set_title(f"{100 * peak:.0f} % of leptons within 1 % of the beam energy, none above it", loc="left")
plt.show()
verdict("A6", above == 0 and peak > 0.5, f"{100 * above:.3g} % above the beam energy; softest lepton {1e3 * El.min():.3g} MeV")

### A7. Constant event weights

The card asks for unweighted events (KEYOPT 3101), so every event must carry the same weight. Then each event
stands for exactly σ/N of cross section, which is what `wx_bhabha.py` assumes. Only the first 10⁵ events are read
from `out.lhe`, which is large.

In [ ]:
w, nxt = [], False
lhe = os.path.join(SAMPLE, "out.lhe")
if os.path.isfile(lhe):
    with open(lhe) as f:
        for line in f:
            if nxt:
                w.append(float(line.split()[2])); nxt = False
                if len(w) >= 100000: break
            elif line.strip() == "<event>": nxt = True
    w = np.array(w)
    verdict("A7", np.ptp(w) <= 1e-12 * abs(w).max(), f"{len(w)} weights, all equal to {w[0]:g}")
else:
    print("no out.lhe: A7 skipped")

### A8. Two independent seeds agree (optional)

Needs a second sample: `run_bhlumi.sh 3 1000000` and `lhe_to_npz.py` on it. **You should see** the ratio of the
two seeds' angular distributions flat at 1 within errors.

In [ ]:
f2 = os.path.join(SAMPLE2, "events.npz")
if os.path.isfile(f2):
    ev2 = np.load(f2)
    edges8 = np.linspace(TH1, TH2, 17); mid8 = 0.5 * (edges8[1:] + edges8[:-1])
    h1 = np.histogram(te, edges8)[0].astype(float); h2 = np.histogram(t0.polar(ev2["ele"], -1), edges8)[0].astype(float)
    r = (h1 / h1.sum()) / (h2 / h2.sum()); er = r * np.sqrt(1 / h1 + 1 / h2)
    fig, ax = plt.subplots()
    ax.axhline(1, color=ORANGE); ax.errorbar(1e3 * mid8, r, er, fmt="o", color=BLUE)
    ax.set_xlabel("θ of e⁻ [mrad]"); ax.set_ylabel("seed 2 / seed 3"); ax.set_title("The two seeds have the same shape", loc="left")
    plt.show()
    x1, x2 = float(ev["xsec_nb"]), float(ev2["xsec_nb"])
    pull = (x1 - x2) / math.hypot(x1 * float(ev["rel_err"]), x2 * float(ev2["rel_err"]))
    chi2 = (((r - 1) / er)**2).sum() / (len(r) - 1)
    verdict("A8", abs(pull) < 3 and chi2 < 2, f"cross sections differ by {pull:+.2f}σ; shape χ²/ndf = {chi2:.2f}")
else:
    print(f"no {f2}: A8 skipped")

## Part B: from BHLUMI to WarpX

### B1 and B5. Momentum and energy are conserved, before and after the hand-off

Adding up every final-state particle of an event must give zero total momentum and the full collision energy
√s = 91.2 GeV. This is checked on the BHLUMI file (B1) and again on the events exactly as `wx_bhabha.py` hands
them to WarpX, after the z-mirroring (B5). **You should see** both histograms far to the left of the orange
tolerance line: the only error is the number of digits printed in the LHE file.

In [ ]:
rng = np.random.default_rng(1)
m = min(20000, n)
e2, p2, ph2, phe2, w_evt, _ = wx.prepare_events(os.path.join(SAMPLE, "events.npz"), m, rng, 1.0)
tot2 = t0.event_sums(e2.copy(), p2, ph2, phe2)
err1 = np.maximum(np.linalg.norm(tot[:, :3], axis=1), np.abs(tot[:, 3] - sqrt_s)) / sqrt_s
err2 = np.maximum(np.linalg.norm(tot2[:, :3], axis=1), np.abs(tot2[:, 3] - sqrt_s)) / sqrt_s
bins = np.linspace(-17, -3, 57)
fig, ax = plt.subplots()
ax.hist(np.log10(np.maximum(err1, 1e-17)), bins, color=BLUE, histtype="step", lw=2, label="BHLUMI file (B1)")
ax.hist(np.log10(np.maximum(err2, 1e-17)), bins, color=BLUE, histtype="stepfilled", alpha=0.35, label="as given to WarpX (B5)")
ax.axvline(-5, color=ORANGE); ax.text(-5.1, 0.92, "tolerance", color=ORANGE, ha="right", transform=ax.get_xaxis_transform())
ax.set_yscale("log"); ax.set_xlabel("log₁₀( largest violation of momentum or energy / √s ) per event"); ax.set_ylabel("events")
ax.set_title("Every event conserves four-momentum to printing precision", loc="left"); ax.legend(fontsize=9, loc="upper left")
plt.show()
verdict("B1", err1.max() < 1e-5, f"worst event {err1.max():.2g}")
verdict("B5", err2.max() < 1e-5, f"worst event {err2.max():.2g} (head-on frame = CM frame, no boost needed)")

### B2. Every particle is on its mass shell

For any real particle, E² − p² = m². For 45 GeV leptons the electron mass is far below the printing precision,
so the mass can only be read off the soft leptons. **You should see** the soft leptons' mass on one flat line.
BHLUMI uses m = 0.5111 MeV, an older value than today's 0.51100 MeV (grey); the 0.02 % difference has no effect,
because WarpX recomputes energies from the momenta with its own mass.

In [ ]:
lep = np.concatenate([ele, pos]); m2 = lep[:, 3]**2 - (lep[:, :3]**2).sum(1)
softl = lep[:, 3] < 1.0
m_gen = math.sqrt(np.median(m2[softl])) if softl.any() else t0.M_E_GEV
off = np.abs(m2 - m_gen**2) / lep[:, 3]**2
fig, ax = plt.subplots()
if softl.any():
    ax.scatter(1e3 * lep[softl, 3], 1e3 * np.sqrt(np.maximum(m2[softl], 0)), s=10, color=BLUE, label="soft leptons")
ax.axhline(1e3 * t0.M_E_GEV, color=GREY, ls="--", lw=1, label="PDG 0.51100 MeV")
ax.axhline(1e3 * m_gen, color=ORANGE, lw=1.5, label=f"fitted {1e3 * m_gen:.4f} MeV")
ax.set_xscale("log"); ax.set_ylim(0.5105, 0.5115); ax.set_xlabel("lepton energy [MeV]"); ax.set_ylabel("√(E² − p²) [MeV]")
ax.set_title("Soft leptons all have the same mass", loc="left"); ax.legend(fontsize=9)
plt.show()
mph = (np.abs((ph[:, 3]**2 - (ph[:, :3]**2).sum(1))) / ph[:, 3]**2).max() if len(ph) else 0.0
verdict("B2", off.max() < 1e-9 and mph < 1e-9 and abs(m_gen / t0.M_E_GEV - 1) < 1e-3,
        f"m = {1e3 * m_gen:.5f} MeV ({100 * (m_gen / t0.M_E_GEV - 1):+.3f} % from PDG); worst lepton {off.max():.2g}, photon {mph:.2g}")

### B3. Momentum becomes WarpX's velocity variable correctly

WarpX tracks u = γv (in m/s), so the code converts p [GeV/c] into u = p/mₑ. For an electron u/c is just p/(mₑc):
a 45.6 GeV electron has u/c = 89 237. **You should see** every lepton on the orange line u/c = p/(mₑc).

In [ ]:
ux, uy, uz = wx.u_from_p(lep[:5000]); u = np.sqrt(ux**2 + uy**2 + uz**2) / wx.CLIGHT
pmag = np.linalg.norm(lep[:5000, :3], axis=1); ref = pmag / t0.M_E_GEV
fig, ax = plt.subplots()
xx = np.array([ref.min(), ref.max()]); ax.plot(xx, xx, color=ORANGE, label="u/c = p/(mₑc)")
ax.scatter(ref, u, s=6, color=BLUE, alpha=0.5, lw=0, label="converted by wx_bhabha.py")
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("p / (mₑc)"); ax.set_ylabel("u / c given to WarpX")
r = np.abs(u / ref - 1).max()
ax.set_title(f"Conversion exact to {r:.0e}", loc="left"); ax.legend(fontsize=9)
plt.show()
verdict("B3", r < 1e-6, f"largest relative difference {r:.2g}")

### B4. Each lepton is sent towards the oncoming bunch

BHLUMI has the e⁻ beam moving along −z; WarpX's electron bunch `beam1` moves along +z. So `wx_bhabha.py` flips
the sign of every p_z. **You should see** the electrons' direction cos θ_z jump from −1 (BHLUMI) to +1 (WarpX).
A handful stay behind: in events where a photon took almost the whole beam energy, the leftover lepton pair
recoils backwards. That is physical, and the test requires every such event to be very hard (v > 0.5).

In [ ]:
pick = np.random.default_rng(1).choice(n, size=m, replace=False)           # the same events prepare_events drew
flip = np.array([1.0, 1.0, -1.0, 1.0])
exact = np.array_equal(e2, ele[pick] * flip) and np.array_equal(p2, pos[pick] * flip)
cz = lambda p4: p4[:, 2] / np.linalg.norm(p4[:, :3], axis=1)
fig, axs = plt.subplots(1, 2, figsize=(8, 3.4), sharey=True)
b = np.linspace(-1, 1, 81)
axs[0].hist(cz(ele[pick]), b, color=GREY, histtype="stepfilled"); axs[0].set_title("BHLUMI: e⁻ moves along −z", loc="left", fontsize=11)
axs[1].hist(cz(e2), b, color=BLUE, histtype="stepfilled"); axs[1].set_title("WarpX: e⁻ moves along +z with beam1", loc="left", fontsize=11)
for ax in axs: ax.set_yscale("log"); ax.set_xlabel("cos θ_z of the electron")
axs[0].set_ylabel("electrons"); plt.tight_layout(); plt.show()

pr = e2 + p2; v2 = 1 - (pr[:, 3]**2 - (pr[:, :3]**2).sum(1)) / s
back = (e2[:, 2] <= 0) | (p2[:, 2] >= 0)
if back.any(): print(f"   {back.sum()} events with a lepton moving backwards; their v = 1 - s'/s:", np.round(np.sort(v2[back]), 4))
verdict("B4", exact and (not back.any() or v2[back].min() > 0.5),
        f"flip exact: {exact}; {back.sum()} backward events, all very hard" if back.any() else f"flip exact: {exact}")

### B6. Crossing angle: the vertex sampler is mirror-symmetric (FCC input)

In WarpX's head-on frame a crossing angle makes the collision point sweep sideways in x as the bunches pass
through each other, so a vertex's x is tied to its time. Reversing the crossing angle must reverse the sweep.
**You should see** two tilted bands that are mirror images in x, and the same luminosity for both.

In [ ]:
c = wx.eval_constants(wx.read_inputs(FCC)); phi = c["rotation_angle"]; res = {}
for sgn in (+1, -1):
    b6 = wx.Bunches(dict(c, rotation_angle=sgn * phi))
    t_, x_, y_, z_, L_, _ = b6.sample(20000, np.random.default_rng(7)); res[sgn] = (x_, wx.CLIGHT * (t_ - b6.tc), L_)
fig, ax = plt.subplots(figsize=(7.5, 3.8))
for sgn, col, lab in ((+1, BLUE, "+φ"), (-1, ORANGE, "−φ")):
    x_, ct_, L_ = res[sgn]; ax.scatter(1e3 * ct_[:3000], 1e6 * x_[:3000], s=3, color=col, alpha=0.4, lw=0)
    end = np.argmin(ct_) ; ax.annotate(f"half crossing {lab}", (1e3 * ct_[end], 1e6 * x_[end]), xytext=(6, 0), textcoords="offset points", color=col, va="center")
ax.set_xlabel("c × (vertex time − bunch-centre crossing time) [mm]"); ax.set_ylabel("vertex x [µm]")
ax.set_title(f"Reversing the crossing angle mirrors the luminous region (φ = {1e3 * phi:.0f} mrad)", loc="left")
plt.show()
q = np.linspace(0.05, 0.95, 19)
asym = np.abs(np.quantile(res[1][0], q) + np.quantile(res[-1][0], 1 - q)).max() / np.std(res[1][0])
dL = abs(res[1][2] / res[-1][2] - 1)
verdict("B6", asym < 0.1 and dL < 0.02, f"mirror mismatch {asym:.3f} σ; luminosities differ by {100 * dL:.2f} %")

## Part C: where and how often collisions happen (head-on benchmark input)

### C6. The luminous region

Bhabha events happen where both bunches are dense, so their vertices follow the product of two Gaussians. For two
identical round bunches that product is again Gaussian, narrower by √2 in every direction, centred where the bunch
centres cross. **You should see** each blue histogram on its orange curve.

In [ ]:
c6 = wx.eval_constants(wx.read_inputs(HEADON)); b = wx.Bunches(c6)
t_, x_, y_, z_, L_bx, _ = b.sample(20000, np.random.default_rng(3))
dt = c6["dt"]
panels = [("x [µm]", 1e6 * x_, 1e6 * b.sx / math.sqrt(2), 0.0),
          ("z [mm]", 1e3 * z_, 1e3 * b.sz / math.sqrt(2), 0.0),
          ("time [WarpX steps]", t_ / dt, b.sz / (math.sqrt(2) * wx.CLIGHT) / dt, b.tc / dt)]
fig, axs = plt.subplots(1, 3, figsize=(10, 3.2))
ok = True
for ax, (lab, d, sig, mu0) in zip(axs, panels):
    bins_ = np.linspace(mu0 - 4 * sig, mu0 + 4 * sig, 41)
    ax.hist(d, bins_, density=True, color=BLUE, alpha=0.85, histtype="stepfilled")
    xx = np.linspace(bins_[0], bins_[-1], 300)
    ax.plot(xx, np.exp(-0.5 * ((xx - mu0) / sig)**2) / (math.sqrt(2 * math.pi) * sig), color=ORANGE)
    ax.set_xlabel(lab); ax.set_yticks([]); ax.set_title(f"σ = {np.std(d):.3g} vs {sig:.3g}", loc="left", fontsize=10)
    ok &= abs(np.std(d) / sig - 1) < 4 / math.sqrt(2 * len(d))
plt.suptitle("Vertices follow the luminous region (orange: beam σ / √2)", x=0.01, ha="left", fontsize=12)
plt.tight_layout(); plt.show()
verdict("C6", ok, f"widths within statistics; mean vertex at step {np.mean(t_) / dt:.0f} of {int(c6['nt'])}")

### C7. Luminosity per crossing

The luminosity decides how many Bhabhas happen per crossing, and so the weight of every injected event. For
head-on round Gaussian bunches it is 𝓛 = N²/(4πσ²). **You should see** the sampler's value (and WarpX's own,
if a run with the `DifferentialLuminosity` diagnostic is available) on the orange line.

In [ ]:
L0 = b.N1 * b.N2 / (4 * math.pi * b.sx * b.sy)
rows = [("vertex sampler (wx_bhabha.py)", L_bx)]
import bhabha_theory as bt
if os.path.isdir(RUN_DIR):
    Lw = bt.luminosity_from_reduced(RUN_DIR, bt.Params(HEADON))
    if Lw: rows.append(("WarpX DifferentialLuminosity", Lw))
fig, ax = plt.subplots(figsize=(7.5, 0.9 + 0.6 * len(rows)))
ax.axvspan(L0 * 0.99, L0 * 1.01, color=ORANGE, alpha=0.12, lw=0); ax.axvline(L0, color=ORANGE)
for i, (lab, val) in enumerate(rows):
    ax.plot(val, i, "o", color=BLUE, ms=8); ax.text(val, i + 0.3, f"{lab}: {100 * (val / L0 - 1):+.2f} %", ha="center", color=INK, fontsize=9)
ax.set_yticks([]); ax.set_ylim(-0.6, len(rows) - 0.2); ax.set_xlabel("luminosity per crossing [m⁻²]")
ax.set_title(f"N²/(4πσ²) = {L0:.4g} m⁻² (orange, band ±1 %)", loc="left"); plt.show()
verdict("C7", all(abs(val / L0 - 1) < 0.01 for _, val in rows), ", ".join(f"{lab} {val:.4g}" for lab, val in rows))

## Summary

In [ ]:
for test, ok, msg in SUMMARY:
    print(f"{'PASS' if ok else 'FAIL'}  {test:3s}  {msg}")
print(f"\n{sum(ok for _, ok, _ in SUMMARY)} of {len(SUMMARY)} passed")